# Week9 - Ensemble Assignment

- create a training and test set with random_state = 3
- create a pipeline to extract new features
- try bagging & boosting algorithms


In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

names = 'https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/names/us_names.csv'

df = pd.read_csv(names)
df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27999 entries, 0 to 27999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    27999 non-null  object
 1   gender  27999 non-null  object
dtypes: object(2)
memory usage: 656.2+ KB


In [2]:
X = df[['name']]
y = df.gender

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

class MyFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit( self, X, y = None ):
        return self

    def transform(self, X, y=None):
        X['first_letter'] = [name[0] for name in X['name']]
        X['last_letter'] = [name[-1] for name in X['name']]

        return X[['first_letter', 'last_letter']].values

pipe = Pipeline([
    ("feature_engineering", MyFeatures()),
#     ("selector_new", FeatureSelector(["daily_trend"])),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

Xmatrix = pipe.fit_transform(X, y)
Xmatrix

<27999x52 sparse matrix of type '<class 'numpy.float64'>'
	with 55998 stored elements in Compressed Sparse Row format>

In [4]:
pipe[1].get_feature_names_out()

array(['x0_a', 'x0_b', 'x0_c', 'x0_d', 'x0_e', 'x0_f', 'x0_g', 'x0_h',
       'x0_i', 'x0_j', 'x0_k', 'x0_l', 'x0_m', 'x0_n', 'x0_o', 'x0_p',
       'x0_q', 'x0_r', 'x0_s', 'x0_t', 'x0_u', 'x0_v', 'x0_w', 'x0_x',
       'x0_y', 'x0_z', 'x1_a', 'x1_b', 'x1_c', 'x1_d', 'x1_e', 'x1_f',
       'x1_g', 'x1_h', 'x1_i', 'x1_j', 'x1_k', 'x1_l', 'x1_m', 'x1_n',
       'x1_o', 'x1_p', 'x1_q', 'x1_r', 'x1_s', 'x1_t', 'x1_u', 'x1_v',
       'x1_w', 'x1_x', 'x1_y', 'x1_z'], dtype=object)

In [5]:
pd.DataFrame(Xmatrix.toarray(), columns=pipe[1].get_feature_names_out(), dtype=int)

,x0_a,x0_b,x0_c,x0_d,x0_e,x0_f,x0_g,x0_h,x0_i,x0_j,...,x1_q,x1_r,x1_s,x1_t,x1_u,x1_v,x1_w,x1_x,x1_y,x1_z
0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27994,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27995,0,0,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
27996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27997,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
#importing necessary header files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report


# Split the data with random state 3
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)


In [7]:
#creating pipeline for extracting new features and evaluating the results on bagging model

# Bagging Model - using decision tree classifer for bagging
bagging_model = BaggingClassifier(base_estimator=DecisionTreeClassifier(), n_estimators=10, random_state=3)
bagging_pipeline = Pipeline([
    ('preprocessing', pipe), #using pipe from given example to extract features
    ('bagging', bagging_model)
])

# Train and Evaluate Bagging Model
bagging_pipeline.fit(X_train, y_train)
y_pred_bagging = bagging_pipeline.predict(X_test)
print("Bagging Model Accuracy:", accuracy_score(y_test, y_pred_bagging))
print(classification_report(y_test, y_pred_bagging))


/Users/rohithkankipati/anaconda3/envs/macos-tensorflow/lib/python3.9/site-packages/sklearn/ensemble/_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


Bagging Model Accuracy: 0.7485714285714286
              precision    recall  f1-score   support

           F       0.74      0.75      0.74      2723
           M       0.76      0.75      0.75      2877

    accuracy                           0.75      5600
   macro avg       0.75      0.75      0.75      5600
weighted avg       0.75      0.75      0.75      5600



In [8]:
#creating pipeline for extracting new features and evaluating the results on boosting model

# Boosting Model - using adaboost classifier for boosting
boosting_model = AdaBoostClassifier(n_estimators=50, random_state=3)
boosting_pipeline = Pipeline([
    ('preprocessing', pipe), #using pipe from given example to extract features
    ('boosting', boosting_model)
])

# Train and Evaluate Boosting Model
boosting_pipeline.fit(X_train, y_train)
y_pred_boosting = boosting_pipeline.predict(X_test)
print("Boosting Model Accuracy:", accuracy_score(y_test, y_pred_boosting))
print(classification_report(y_test, y_pred_boosting))

Boosting Model Accuracy: 0.7469642857142857
              precision    recall  f1-score   support

           F       0.73      0.75      0.74      2723
           M       0.76      0.74      0.75      2877

    accuracy                           0.75      5600
   macro avg       0.75      0.75      0.75      5600
weighted avg       0.75      0.75      0.75      5600



from this approach we can see that bagging model performs better than the boosting model.